# Two-Bunch BOED — Simulator Evaluation

Evaluates the amortized BOED generator on the two-bunch simulator with known true parameters.

Protocol:
1. Grid scan: `grid_steps` evenly-spaced gunphase measurements (handled internally by the generator).
2. BOED loop: posterior-sampled designs from traced round models. Normalization is fitted automatically from the grid scan on the first BOED call.
3. Diagnostic plot: measurements + T0 posterior + posterior predictive.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
model_dir = "aboed"  # directory with traced .pt models
design_range = [-25.0, 45.0]  # [t_min, t_max] in physical gunphase degrees
observable_name = "charge"
variable_name = "gunphase"
max_measure = 100  # total measurements (grid + BOED)

# True parameters for simulation
true_t0 = 25.0
true_nuisance = [3.0, 0.5, 5.0, 50.0, 1.0, 0.002, 0.0, 0.5, 0.1, 10.0]
# [A, f1, tau1, tau2, tau_kick, sigma, y_shift, irf_sigma, alpha_kick, delta_T]

n_posterior_samples = 2000
n_predictive_curves = 300

In [ ]:
import torch

In [ ]:
from autonomous_control.facet.two_bunch_boed_utils import (
    TwoBunchDoubleExp,
)

In [ ]:
# ── Simulator evaluator ───────────────────────────────────────────────────────

simulator = TwoBunchDoubleExp(noise_type="gaussian")
true_theta_t = torch.tensor([[true_t0]], dtype=torch.float32)
true_nuisance_t = torch.tensor([true_nuisance], dtype=torch.float32)


class SimulatorEnvironment:
    variables = {"gunphase": []}

    def __init__(self, simulator, true_theta_t, true_nuisance_t):
        self.simulator = simulator
        self.true_theta_t = true_theta_t
        self.true_nuisance_t = true_nuisance_t

        self._curr_phase = 0

    def get_variables(self, names):
        return {variable_name: self._curr_phase}

    def set_variables(self, variables):
        self._curr_phase = variables[variable_name]
        return self._curr_phase

    def get_observables(self, names):
        t = torch.tensor([[self._curr_phase]], dtype=torch.float32)
        y = self.simulator(t, self.true_theta_t, self.true_nuisance_t).item()
        return {observable_name: y}


env = SimulatorEnvironment(simulator, true_theta_t, true_nuisance_t)

In [ ]:
from auto_schottky import run_automatic_schottky_scan

X = run_automatic_schottky_scan(env)